# Figure 3 — SPECTER2 semantic-map derived outputs

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'requirements.txt').exists() and (p/'data').is_dir(): return p
    raise RuntimeError('Repository root not found. Run from inside the cloned repository.')
ROOT=find_repo_root()
print('Repository root:', ROOT)

In [ ]:
import matplotlib.pyplot as plt
corpus=pd.read_csv(ROOT/'data/Figure1/Figure1_Corpus_DERIVED_14366.csv')
ledger=pd.read_csv(ROOT/'data/Figure2/Figure2_ScreeningLedger_DERIVED_765.csv')
seeds=pd.read_csv(ROOT/'data/Figure2/Figure2_PositiveSeeds_DERIVED_2.csv')
um=pd.read_csv(ROOT/'data/Figure3/Figure3_UMAP_Cluster_DERIVED_14366.csv')
rr=pd.read_csv(ROOT/'data/Figure3/Figure3_ResidualRanking_DERIVED_12146.csv')
aud=pd.read_csv(ROOT/'data/Figure3/Figure3_Top50Audit_DERIVED.csv')
kqc=pd.read_csv(ROOT/'data/Figure3/Figure3_ClusterK_QC_DERIVED.csv')
assert len(corpus)==14366 and int(corpus.has_abstract.sum())==12913 and int((corpus.has_abstract==0).sum())==1453
assert len(ledger)==765 and len(seeds)==2 and 765+2==767 and int(ledger.Label_numeric.sum())+2==52
assert len(um)==14366 and um.Broad_ID.nunique()==14366
assert int(um.Candidate52.sum())==52
assert um.Cluster_ID.nunique()==6
assert len(rr)==12146 and rr['Residual rank'].tolist()==list(range(1,12147))
assert int(rr['Top-50 audit flag'].sum())==50
assert len(aud)==50 and aud.human_decision.eq('Not relevant').all()
assert int((aud.exclusion_reason=='Non-original publication').sum())==44
assert int((aud.exclusion_reason=='No aging/senescence axis').sum())==6
expected=rr.iloc[:50].Broad_ID.astype(str).tolist(); observed=aud.sort_values('specter2_residual_rank').Broad_ID.astype(str).tolist(); assert expected==observed
selected=kqc.loc[kqc.Selected.astype(str).str.upper().eq('YES'),'k'].astype(int).tolist(); assert selected==[6]
composition=um.groupby(['Cluster_ID','Semantic_domain'],as_index=False).agg(Broad_corpus_n=('Broad_ID','size'),Candidate_n=('Candidate52','sum'))
composition['Broad_corpus_pct']=composition.Broad_corpus_n/14366*100; composition['Candidate_pct']=composition.Candidate_n/52*100
fig,ax=plt.subplots(figsize=(6,5));
for cid,g in um.groupby('Cluster_ID'): ax.scatter(g.UMAP1,g.UMAP2,s=1,alpha=.25,label=str(cid))
plt.close(fig)
print('Figure 3 derived-output validation: PASS')
print({'broad_corpus':14366,'title_abstract':12913,'title_only':1453,'asreview_labeled':767,'candidates':52,'residual':12146,'top50':50,'additional_relevant':0,'clusters':6})
print(composition.to_string(index=False))
print('Note: raw SPECTER2 input, embeddings, and .asreview project are intentionally excluded from the public SLIM repository.')